<a href="https://colab.research.google.com/github/rodrigorissettoterra/SENAI_Concepcao_e_Design_de_ML/blob/main/Dados_Hepatite_Ativ_02.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

IMPORTAÇÃO DAS BIBLIOTECAS

In [1]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.feature_selection import SelectKBest, chi2
from sklearn.decomposition import PCA
from sklearn.svm import SVC
from sklearn.metrics import balanced_accuracy_score

CARREGAMENTO DA BASE

In [2]:
url = "https://www.openml.org/data/get_csv/55/dataset_55_hepatitis.arff"
df = pd.read_csv(url)

TRATAMENTO DE VALORES FALTANTES E TIPOS

In [3]:
df = df.replace('?', np.nan)

Definir colunas categóricas e numéricas

In [4]:
categorical_cols = [
    'SEX', 'STEROID', 'ANTIVIRALS', 'FATIGUE', 'MALAISE', 'ANOREXIA',
    'LIVER_BIG', 'LIVER_FIRM', 'SPLEEN_PALPABLE', 'SPIDERS',
    'ASCITES', 'VARICES', 'HISTOLOGY', 'Class'
]

numeric_cols = [
    'AGE', 'BILIRUBIN', 'ALK_PHOSPHATE', 'SGOT', 'ALBUMIN', 'PROTIME'
]

Converter colunas numéricas e categóricas

In [5]:
for col in numeric_cols:
    df[col] = pd.to_numeric(df[col], errors='coerce')

for col in categorical_cols:
    df[col] = df[col].astype('category')

Preencher valores faltantes

In [6]:
for col in numeric_cols:
    df[col] = df[col].fillna(df[col].median())

for col in categorical_cols:
    df[col] = df[col].fillna(df[col].mode()[0])

Remover duplicados

In [7]:
df = df.drop_duplicates()

TRANSFORMAÇÃO DAS VARIÁVEIS CATEGÓRICAS

One-hot encoding

In [8]:
df = pd.get_dummies(df, columns=categorical_cols, drop_first=True)

DEFINIÇÃO DE X E y

In [9]:
y = df['Class_LIVE']
X = df.drop(columns=['Class_LIVE'])

DIVISÃO TREINO E TESTE

In [10]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

CENÁRIO 1 - SELEÇÃO POR CORRELAÇÃO

In [11]:
# Calcular correlação absoluta com o target
correlations = X_train.corrwith(y_train.astype(int)).abs()

# Selecionar as 12 melhores variáveis
top12_corr = correlations.sort_values(ascending=False).head(12).index

X_train_corr = X_train[top12_corr]
X_test_corr = X_test[top12_corr]

# Padronizar
scaler_corr = StandardScaler()
X_train_corr_scaled = scaler_corr.fit_transform(X_train_corr)
X_test_corr_scaled = scaler_corr.transform(X_test_corr)

CENÁRIO 2 - SELEÇÃO POR CHI-QUADRADO

In [12]:
scaler_chi = MinMaxScaler()
X_train_chi_base = scaler_chi.fit_transform(X_train)
X_test_chi_base = scaler_chi.transform(X_test)

selector_chi = SelectKBest(score_func=chi2, k=12)
X_train_chi = selector_chi.fit_transform(X_train_chi_base, y_train.astype(int))
X_test_chi = selector_chi.transform(X_test_chi_base)

CENÁRIO 3 - PCA COM 12 COMPONENTES

In [13]:
scaler_pca = StandardScaler()
X_train_pca_base = scaler_pca.fit_transform(X_train)
X_test_pca_base = scaler_pca.transform(X_test)

pca = PCA(n_components=12)
X_train_pca = pca.fit_transform(X_train_pca_base)
X_test_pca = pca.transform(X_test_pca_base)

MODELO SVM COM KERNEL RBF

In [14]:
model = SVC(kernel='rbf', class_weight='balanced', random_state=42)

TREINAMENTO E AVALIAÇÃO

In [15]:
# Cenário 1 - Correlação
model.fit(X_train_corr_scaled, y_train)
y_pred_corr = model.predict(X_test_corr_scaled)
acc_corr = balanced_accuracy_score(y_test, y_pred_corr)

# Cenário 2 - Chi²
model.fit(X_train_chi, y_train)
y_pred_chi = model.predict(X_test_chi)
acc_chi = balanced_accuracy_score(y_test, y_pred_chi)

# Cenário 3 - PCA
model.fit(X_train_pca, y_train)
y_pred_pca = model.predict(X_test_pca)
acc_pca = balanced_accuracy_score(y_test, y_pred_pca)

RESULTADOS

In [16]:
print("Distribuição do target:")
print(y.value_counts())
print("\nResultados finais:")
print(f"Correlação: {acc_corr:.4f}")
print(f"Chi²:       {acc_chi:.4f}")
print(f"PCA:        {acc_pca:.4f}")

Distribuição do target:
Class_LIVE
True     123
False     32
Name: count, dtype: int64

Resultados finais:
Correlação: 0.6500
Chi²:       0.8400
PCA:        0.7333


O melhor desempenho do método qui-quadrado sugere que o problema possui forte dependência entre variáveis categóricas e o desfecho clínico, indicando que relações não lineares e discretas são mais relevantes do que relações lineares simples ou transformações por redução de dimensionalidade.

Foram avaliadas três abordagens de redução de dimensionalidade: seleção por correlação, seleção por qui-quadrado e PCA. O modelo SVM com kernel RBF apresentou melhor desempenho no cenário baseado em qui-quadrado (0.84), seguido por PCA (0.73) e correlação (0.65). Os resultados indicam que as variáveis categóricas possuem maior relevância para o problema, e que a dependência estatística entre atributos e o desfecho clínico é mais significativa do que relações lineares. Além disso, a utilização da acurácia balanceada mostrou-se adequada devido ao desbalanceamento entre as classes, garantindo uma avaliação mais justa do modelo.

A superioridade do método qui-quadrado também sugere que técnicas baseadas em seleção de atributos podem ser mais eficazes do que técnicas de transformação, como PCA, quando a interpretabilidade e a relevância clínica dos atributos são importantes.